In [1]:
# ==============================================================================
# PHASE 1: INSTALLATION
# ==============================================================================
import torch, os

if not torch.cuda.is_available():
    raise RuntimeError("❌ NO GPU! Select T4 GPU in Runtime -> Change runtime type.")

print("⏳ Installing Video Studio...")
!pip uninstall -y diffusers transformers huggingface_hub
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers==0.25.1 transformers==4.37.2 huggingface_hub==0.23.0 accelerate==0.28.0 peft==0.8.2
!pip install -q "moviepy<2.0" opencv-python gtts imageio-ffmpeg
!apt-get install -y ffmpeg

print("✅ Installation Complete.")

⏳ Installing Video Studio...
Found existing installation: diffusers 0.36.0
Uninstalling diffusers-0.36.0:
  Successfully uninstalled diffusers-0.36.0
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 36.2 MB/s eta 0:00:00


In [2]:
# ==============================================================================
# PHASE 2: DYNAMIC CONFIG & ADVANCED SCRIPT
# ==============================================================================
import os

WORKSPACE = "/content/space_cat_studio"
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)
for d in ["shots", "audio", "temp"]: os.makedirs(d, exist_ok=True)

# 🎛️ DYNAMIC SETTINGS (EDIT THIS!)
USER_CONFIG = {
    "ratio": "9:16",         # "9:16" (Shorts), "16:9" (YouTube), "1:1" (Insta)
    "fps": 24,               # 24 (Cinema) or 30 (TV)
    "style": "pixar_3d",     # "pixar_3d", "anime", "realistic"
}

# 📐 Auto-Calculate Resolution
if USER_CONFIG["ratio"] == "16:9":
    W, H = 896, 512
elif USER_CONFIG["ratio"] == "9:16":
    W, H = 512, 896
else:
    W, H = 512, 512

CONFIG = {
    "width": W,
    "height": H,
    "fps": USER_CONFIG["fps"],
}

# 🎨 MASTER PROMPTS (The Secret Sauce)
STYLES = {
    "pixar_3d": "3d disney pixar style, unreal engine 5 render, octane render, volumetric lighting, bloom, ultra detailed fur, 8k, cinematic depth of field",
    "anime": "studio ghibli style, makoto shinkai, anime screencap, vibrant colors, detailed background, 4k, cel shaded",
    "realistic": "National Geographic documentary footage, 8k, hyperrealistic, raw photo, cinematic lighting, shot on ARRI Alexa, sharp focus"
}

NEGATIVE_PROMPT = (
    "deformed, distorted, disfigured, doll, bad anatomy, bad eyes, crossed eyes, "
    "low quality, worst quality, grainy, blurry, text, watermark, signature, "
    "mutation, extra limb, ugly, jpeg artifacts, low res, glitch"
)

# 🎬 THE SCRIPT (Dual-Shot Logic)
# Each scene has 2 shots to ensure ~4-5 seconds of unique forward video.
SCENES = [
    {
        "id": 1,
        "text": "Meet Captain Whiskers, the bravest explorer in the galaxy!",
        "shots": [
            {
                "subject": "anthropomorphic ginger cat astronaut putting on high-tech helmet",
                "action": "preparing for launch, determined expression",
                "env": "futuristic bright white spaceship cockpit",
                "cam": "cinematic close up, shallow depth of field"
            },
            {
                "subject": "cat astronaut sitting in pilot chair",
                "action": "pushing glowing buttons, console lights blinking",
                "env": "stars moving fast outside window",
                "cam": "medium shot, side angle"
            }
        ]
    },
    {
        "id": 2,
        "text": "His ship zooms past colorful candy planets.",
        "shots": [
            {
                "subject": "sleek white spaceship",
                "action": "launching engines with blue fire",
                "env": "deep space background",
                "cam": "wide shot, camera shake"
            },
            {
                "subject": "spaceship",
                "action": "flying past giant pink donut planet",
                "env": "galaxy made of glowing neon candies and nebula",
                "cam": "dynamic tracking shot"
            }
        ]
    },
    {
        "id": 3,
        "text": "Suddenly, he spots a moon made entirely of cheese!",
        "shots": [
            {
                "subject": "view from inside cockpit",
                "action": "looking out window",
                "env": "giant yellow moon appearing in distance",
                "cam": "pov shot"
            },
            {
                "subject": "giant yellow moon with cheese texture",
                "action": "floating in space, glowing softly",
                "env": "dark space with sparkling stars",
                "cam": "epic pan shot"
            }
        ]
    },
    {
        "id": 4,
        "text": "He lands in a magical forest of giant lollipops.",
        "shots": [
            {
                "subject": "spaceship landing gear",
                "action": "touching down on purple grass, dust kicking up",
                "env": "alien planet surface",
                "cam": "ground level shot"
            },
            {
                "subject": "cat astronaut",
                "action": "walking curiously looking up",
                "env": "forest of giant glowing lollipop trees",
                "cam": "dolly forward"
            }
        ]
    },
    {
        "id": 5,
        "text": "A friendly robot greets him with a wave.",
        "shots": [
            {
                "subject": "cute round white robot with blue eyes",
                "action": "rolling forward happily",
                "env": "alien landscape sunset",
                "cam": "low angle"
            },
            {
                "subject": "robot and cat astronaut",
                "action": "waving hello to each other",
                "env": "golden hour lighting",
                "cam": "two shot, medium"
            }
        ]
    },
    {
        "id": 6,
        "text": "Together, they watch the shooting stars.",
        "shots": [
            {
                "subject": "cat and robot silhouettes",
                "action": "sitting side by side on a hill",
                "env": "night sky",
                "cam": "wide back shot"
            },
            {
                "subject": "night sky",
                "action": "exploding with colorful fireworks and shooting stars",
                "env": "beautiful nebula background",
                "cam": "tilt up"
            }
        ]
    }
]

print(f"✅ Config Loaded: {USER_CONFIG['style']} | {USER_CONFIG['ratio']} | {CONFIG['fps']} FPS")

✅ Config Loaded: pixar_3d | 9:16 | 24 FPS


In [3]:
# ==============================================================================
# PHASE 3: GENERATION (Best Quality + Safe Mode)
# ==============================================================================
import torch, gc, os, subprocess, shutil
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler, AutoencoderKL
from diffusers.utils import export_to_video

print("⚡ Loading Director's Engine...")

# Load AI
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=torch.float16)
pipe = AnimateDiffPipeline.from_pretrained("Lykon/DreamShaper", motion_adapter=adapter, vae=vae, torch_dtype=torch.float16)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, beta_schedule="linear", steps_offset=1, clip_sample=False)
pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

generated_shots = []
master_style = STYLES[USER_CONFIG["style"]]

for scene in SCENES:
    print(f"\n🎬 Filming Scene {scene['id']}...")
    scene_clips = []

    for i, shot in enumerate(scene['shots']):
        shot_id = f"{scene['id']}_{i}"
        print(f"   🎥 Take {i+1}: {shot['cam']}...")

        # 🧠 PROMPT INJECTION
        full_prompt = (
            f"{master_style}, "
            f"{shot['subject']}, {shot['action']}, {shot['env']}, "
            f"{shot['cam']}, masterpiece, best quality, 8k, vivid colors, high motion"
        )

        # GENERATE
        output = pipe(
            prompt=full_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_frames=16,
            guidance_scale=8.5,     # High adherence to prompt
            num_inference_steps=35, # High quality steps
            width=CONFIG['width'],
            height=CONFIG['height'],
        )

        # SAVE & SMOOTH
        raw_path = f"shots/raw_{shot_id}.mp4"
        smooth_path = f"shots/smooth_{shot_id}.mp4"
        export_to_video(output.frames[0], raw_path, fps=8)

        # Safe FFmpeg Smoothing
        cmd = [
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-i", raw_path,
            "-filter:v", f"minterpolate=fps={CONFIG['fps']}:mi_mode=mci",
            smooth_path
        ]
        subprocess.run(cmd)

        # Fallback Check
        if os.path.exists(smooth_path) and os.path.getsize(smooth_path) > 1000:
            scene_clips.append(smooth_path)
        else:
            print(f"   ⚠️ Smoothing failed. Using raw.")
            shutil.copy(raw_path, smooth_path)
            scene_clips.append(smooth_path)

        torch.cuda.empty_cache()
        gc.collect()

    generated_shots.append(scene_clips)

del pipe, adapter, vae
torch.cuda.empty_cache()
gc.collect()
print("✅ Phase 3 Complete.")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/utils/outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


⚡ Loading Director's Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...


🎬 Filming Scene 6...
   🎥 Take 1: wide back shot...


  0%|          | 0/35 [00:00<?, ?it/s]

   🎥 Take 2: tilt up...


  0%|          | 0/35 [00:00<?, ?it/s]

✅ Phase 3 Complete.


In [4]:
# ==============================================================================
# PHASE 4: AUDIO ASSETS
# ==============================================================================
import numpy as np
from scipy.io.wavfile import write
from gtts import gTTS

print("🎙️ Recording Voiceovers...")
voice_files = []
for scene in SCENES:
    tts = gTTS(scene['text'], lang='en', tld='co.uk')
    path = f"audio/voice_{scene['id']}.mp3"
    tts.save(path)
    voice_files.append(path)

print("🎹 Synthesizing Music...")
def generate_music(duration=60):
    sr = 44100
    t = np.linspace(0, duration, int(sr * duration), False)
    melody = np.zeros_like(t)
    notes = [523.25, 659.25, 783.99, 880.00]
    bpm = 110
    note_len = 60 / bpm
    current_time = 0
    idx = 0
    while current_time < duration:
        freq = notes[idx % 4]
        if idx % 8 == 7: freq *= 1.5
        start = int(current_time * sr)
        end = int((current_time + note_len) * sr)
        if end > len(t): break
        t_note = np.linspace(0, note_len, end - start)
        melody[start:end] += np.sin(2 * np.pi * freq * t_note) * np.exp(-5 * t_note)
        current_time += note_len
        idx += 1
    return melody * 0.5

music_data = generate_music()
write("audio/bg_music.wav", 44100, (music_data * 32767).astype(np.int16))
print("✅ Phase 4 Complete.")

🎙️ Recording Voiceovers...
🎹 Synthesizing Music...
✅ Phase 4 Complete.


In [5]:
# ==============================================================================
# PHASE 5: FINAL ASSEMBLY (Director's Cut)
# ==============================================================================
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips, CompositeAudioClip

print("🎞️ Assembling Final Cut...")

final_scene_clips = []

for i, shots in enumerate(generated_shots):
    # Verify files
    if len(shots) < 2: continue

    clip_a = VideoFileClip(shots[0])
    clip_b = VideoFileClip(shots[1])

    # Crossfade Join (Smooth Transition)
    combined = concatenate_videoclips([clip_a, clip_b], method="compose", padding=-0.5)

    # Sync Audio
    if i < len(voice_files) and os.path.exists(voice_files[i]):
        voice = AudioFileClip(voice_files[i])

        # Stretch video slightly if voice is long
        if voice.duration > combined.duration:
            ratio = combined.duration / (voice.duration + 0.5)
            combined = combined.speedx(ratio)

        voice_start = (combined.duration - voice.duration) / 2
        voice = voice.set_start(max(0, voice_start))
        combined = combined.set_audio(CompositeAudioClip([voice]))

    final_scene_clips.append(combined)

# Final Render
if not final_scene_clips: raise RuntimeError("No clips found!")

main_video = concatenate_videoclips(final_scene_clips, method="compose")
bg_music = AudioFileClip("audio/bg_music.wav").volumex(0.15)
bg_music = bg_music.subclip(0, main_video.duration)
final_mix = CompositeAudioClip([main_video.audio, bg_music])
main_video = main_video.set_audio(final_mix)

output_file = f"DirectorCut_{USER_CONFIG['style']}.mp4"
print(f"🚀 Rendering {output_file}...")

main_video.write_videofile(
    output_file,
    fps=CONFIG['fps'],
    codec="libx264",
    audio_codec="aac",
    preset="medium",
    threads=2
)

from google.colab import files
files.download(output_file)
print("✨ DONE!")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



🎞️ Assembling Final Cut...
🚀 Rendering DirectorCut_pixar_3d.mp4...
Moviepy - Building video DirectorCut_pixar_3d.mp4.
MoviePy - Writing audio in DirectorCut_pixar_3dTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video DirectorCut_pixar_3d.mp4



Moviepy - Done !
Moviepy - video ready DirectorCut_pixar_3d.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✨ DONE!
